# 00 - Setup: Carga de Dados no SQL Server

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **SeguroDB** no SQL Server 2025.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessível na porta `1433`
- Driver ODBC 18 instalado (`sudo apt install unixodbc-dev`)

## 1. Configuração e Conexão

In [1]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

Servidor: localhost:1433
Database: InsuranceDB


In [2]:
# Conexão ao SQL Server (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

Conectado ao SQL Server (master) com sucesso!


## 2. Criar Database SeguroDB

In [3]:
# Criar o database se não existir
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

Database [InsuranceDB] criado/verificado com sucesso!


In [4]:
# Conectar ao database SeguroDB
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

Conectado ao [InsuranceDB] com sucesso!


## 3. Criar Tabelas

In [6]:
# DDL - Criação das tabelas
ddl_statements = [
    # Tabelas de domínio (sem FK)
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Customers')   
    CREATE TABLE Customers (
        customer_id INT IDENTITY(1,1) PRIMARY KEY,
        first_name NVARCHAR(50) NOT NULL,
        last_name NVARCHAR(50) NOT NULL,
        date_of_birth DATE,
        email NVARCHAR(100) UNIQUE,
        phone NVARCHAR(20),
        address NVARCHAR(255)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Insurance_Types')
    CREATE TABLE Insurance_Types (
        insurance_type_id INT IDENTITY(1,1) PRIMARY KEY,
        type_name NVARCHAR(50) NOT NULL,
        description NVARCHAR(MAX)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Policies')
    CREATE TABLE Policies (
        policy_id INT IDENTITY(1,1) PRIMARY KEY,
        customer_id INT NOT NULL,
        insurance_type_id INT NOT NULL,
        policy_number NVARCHAR(50) NOT NULL UNIQUE,
        start_date DATE NOT NULL,
        end_date DATE NOT NULL,
        premium_amount DECIMAL(10,2) NOT NULL,
        policy_status NVARCHAR(20),

        CONSTRAINT FK_Policies_Customers
            FOREIGN KEY (customer_id)
            REFERENCES Customers(customer_id),

        CONSTRAINT FK_Policies_InsuranceTypes
            FOREIGN KEY (insurance_type_id)
            REFERENCES Insurance_Types(insurance_type_id)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Claims')
    CREATE TABLE Claims (
        claim_id INT IDENTITY(1,1) PRIMARY KEY,
        policy_id INT NOT NULL,
        claim_date DATE NOT NULL,
        claim_amount DECIMAL(10,2),
        claim_status NVARCHAR(20),
        description NVARCHAR(MAX),

        CONSTRAINT FK_Claims_Policies
            FOREIGN KEY (policy_id)
            REFERENCES Policies(policy_id)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Payments')
    CREATE TABLE Payments (
        payment_id INT IDENTITY(1,1) PRIMARY KEY,
        policy_id INT NOT NULL,
        payment_date DATE NOT NULL,
        amount DECIMAL(10,2) NOT NULL,
        payment_method NVARCHAR(30),

        CONSTRAINT FK_Payments_Policies
            FOREIGN KEY (policy_id)
            REFERENCES Policies(policy_id)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Agents')
    CREATE TABLE Agents (
        agent_id INT IDENTITY(1,1) PRIMARY KEY,
        first_name NVARCHAR(50) NOT NULL,
        last_name NVARCHAR(50) NOT NULL,
        email NVARCHAR(100) UNIQUE,
        phone NVARCHAR(20)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'Agent_Policies')
    CREATE TABLE Agent_Policies (
        agent_policy_id INT IDENTITY(1,1) PRIMARY KEY,
        agent_id INT NOT NULL,
        policy_id INT NOT NULL,
        role NVARCHAR(30),
        assigned_date DATE,

        CONSTRAINT FK_AgentPolicies_Agents
            FOREIGN KEY (agent_id)
            REFERENCES Agents(agent_id),

        CONSTRAINT FK_AgentPolicies_Policies
            FOREIGN KEY (policy_id)
            REFERENCES Policies(policy_id),

        CONSTRAINT UQ_AgentPolicies UNIQUE (agent_id, policy_id)
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)
    
print('Todas as tabelas foram criadas com sucesso!')

Todas as tabelas foram criadas com sucesso!


## 4. Carregar Dados dos CSVs

In [9]:
# Ordem de carga (respeitar dependências)
tabelas = [
    'Customers', 'Insurance_Types', 'Policies', 'Claims',
    'Payments', 'Agents', 'Agent_Policies'
]

data_dir = os.path.join(os.getcwd(), '../data')

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')
    
    # Verificar se a tabela já tem dados
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    
    if count > 0:
        print(f'  ⏭️  {tabela}: já contém {count} registros, pulando...')
        continue
    
    # Ler CSV
    df = pd.read_csv(csv_path)
    
    # Limpar espaços em colunas string
    for col in df.select_dtypes(include=['str', 'object']).columns:
        df[col] = df[col].str.strip()
    
    # Inserir dados via executemany
    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    
    # Converter NaN para None
    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]
    
    # Inserir em lotes de 1000
    batch_size = 2000
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cursor.executemany(insert_sql, batch)
    
    print(f'  ✅ {tabela}: {len(data)} registros inseridos')

print('\n🎉 Carga de dados concluída!')

  ✅ Customers: 10 registros inseridos
  ✅ Insurance_Types: 10 registros inseridos
  ✅ Policies: 10 registros inseridos
  ✅ Claims: 10 registros inseridos
  ✅ Payments: 10 registros inseridos
  ✅ Agents: 10 registros inseridos
  ✅ Agent_Policies: 10 registros inseridos

🎉 Carga de dados concluída!


## 5. Validação

In [10]:
# Verificar contagem de registros em cada tabela
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<20} {count:>10}')

print('\n✅ Validação concluída!')

Tabela                Registros
--------------------------------
Customers                    10
Insurance_Types              10
Policies                     10
Claims                       10
Payments                     10
Agents                       10
Agent_Policies               10

✅ Validação concluída!


In [12]:
# Amostra de dados de algumas tabelas
for tabela in ['Customers', 'Insurance_Types', 'Policies']:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
    print(df_sample.to_string(index=False))


--- CUSTOMERS (primeiros 5 registros) ---
 customer_id first_name last_name date_of_birth                email    phone      address
           1       John       Doe    1985-01-10   john.doe@email.com 555-1111  123 Main St
           2       Jane     Smith    1990-02-20 jane.smith@email.com 555-1112   456 Oak St
           3     Carlos     Silva    1978-03-15   carlos.s@email.com 555-1113  789 Pine St
           4       Anna     Brown    1988-04-18     anna.b@email.com 555-1114 321 Maple St
           5       Mike    Wilson    1992-05-22     mike.w@email.com 555-1115 654 Cedar St

--- INSURANCE_TYPES (primeiros 5 registros) ---
 insurance_type_id type_name          description
                 1      Auto Automobile insurance
                 2    Health     Health insurance
                 3      Life       Life insurance
                 4      Home Homeowners insurance
                 5    Travel     Travel insurance

--- POLICIES (primeiros 5 registros) ---
 policy_id  customer

/tmp/ipykernel_19487/244899174.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_19487/244899174.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)
/tmp/ipykernel_19487/244899174.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql(f'SELECT TOP 5 * FROM {tabela}', conn)


In [13]:
# Encerrar conexão
cursor.close()
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
